# ADM NB Explained

This notebook shows exactly how all the values in an ADM model report
are calculated. It also shows how the propensity is calculated for a
particular customer.

We use one of the shipped datamart exports for the example. This is a
model very similar to one used in some of the ADM PowerPoint/Excel deep
dive examples. You can change this notebook to apply to your own data.


> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm`, `healthcheck` extras. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm,healthcheck]"`.


In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately

import plotly.io as pio

pio.renderers.default = "notebook_connected"


In [2]:
import polars as pl
import numpy as np

import plotly.express as px
from math import log
from great_tables import GT
from pdstools import datasets
from pdstools.utils import cdh_utils

In [3]:
model_name = "AutoNew84Months"
predictor_name = "IH.Email.Outbound.Accepted.pyHistoricalOutcomeCount"
channel = "Web"

For the example we pick one particular model over a channel.
To explain the ADM model report, we use one of the active predictors as an
example. Swap for any other predictor when using different data.

In [4]:
dm = datasets.cdh_sample()

model = dm.combined_data.filter(
    (pl.col("Name") == model_name) & (pl.col("Channel") == channel)
)

modelpredictors = (
    dm.combined_data.join(
        model.select(pl.col("ModelID").unique()), on="ModelID", how="inner"
    )
    .filter(pl.col("EntryType") != "Inactive")
    .with_columns(
        Action=pl.concat_str(["Issue", "Group"], separator="/"),
        PredictorName=pl.col("PredictorName").cast(pl.Utf8),
    )
    .collect()
)

predictorbinning = modelpredictors.filter(
    pl.col("PredictorName") == predictor_name
).sort("BinIndex")

In [5]:
model_id = None

if (modelpredictors.select(pl.col("ModelID").unique()).shape[0] > 1) and (
    model_id is None
):
    display(
        model.group_by("ModelID")
        .agg(
            number_of_predictors=pl.col("PredictorName").n_unique(),
            model_performance=cdh_utils.weighted_performance_polars() * 100,
            response_count=pl.sum("ResponseCount"),
        )
        .collect()
    )
    raise Exception(
        f"**{model_name}** model has multiple instances."
        "\nThis could be due to the same model name being used in different configurations, directions, issues, or having multiple treatments."
        "\nTo ensure the selection of a unique model, please choose a model_id from the table above and update the `model_id` variable at the top of this cell."
        "\nAfterward, rerun this cell."
        f"\nSee model IDs in {model_name} model above:"
    )
if model_id is not None:
    if (
        model_id
        not in modelpredictors.select(pl.col("ModelID").unique())
        .get_column("ModelID")
        .to_list()
    ):
        raise Exception(
            f"The {model_name} model does not have a model ID: {model_id}."
            f"Please ensure that the spelling of the model ID is correct."
            f"You can run `modelpredictors.select(pl.col('ModelID').unique().implode()).row(0)` to see the exact spellings of your IDs."
            "After updating the `model_id`, you can restart the notebook from the beginning."
        )

    predictors_in_selected_model = (
        modelpredictors.filter(pl.col("ModelID") == model_id)
        .select(pl.col("PredictorName").unique())
        .get_column("PredictorName")
        .to_list()
    )
    if predictor_name not in predictors_in_selected_model:
        raise Exception(
            f"{predictor_name} is not a predictor of the model with ID: {model_id}."
            "Please choose one of the available predictors below and update the **predictor_name** variable in the cell above:"
            f"\nAvailable Predictors:\n{predictors_in_selected_model}."
        )

    modelpredictors = modelpredictors.filter(pl.col("ModelID") == model_id)
    predictorbinning = predictorbinning.filter(pl.col("ModelID") == model_id)
    print(f"{model_name} model with **{model_id}** model ID is selected successfully.")

## Model Overview

The selected model is shown below. Only the currently active predictors are used for the propensity calculation, so only showing those.



In [6]:
from great_tables import loc, style

GT(
    modelpredictors.select(
        pl.col("Action").unique(),
        pl.col("Channel").unique(),
        pl.col("Name").unique(),
        pl.col("PredictorName")
        .unique()
        .sort()
        .implode()
        .list.join(", ")
        .alias("Active Predictors"),
        (pl.col("Performance").unique() * 100).alias("Model Performance (AUC)"),
    ).transpose(include_header=True)
).tab_header("Overview").tab_options(column_labels_hidden=True).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="column")
)

GT(_tbl_data=shape: (5, 2)
┌─────────────────────────┬─────────────────────────────────┐
│ column                  ┆ column_0                        │
│ ---                     ┆ ---                             │
│ str                     ┆ str                             │
╞═════════════════════════╪═════════════════════════════════╡
│ Action                  ┆ Sales/AutoLoans                 │
│ Channel                 ┆ Web                             │
│ Name                    ┆ AutoNew84Months                 │
│ Active Predictors       ┆ Classifier, Customer.Age, Cust… │
│ Model Performance (AUC) ┆ 77.4901                         │
└─────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x75f62accc490>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='column_0', type=<ColInfoTypeEnum.default: 1>, column_label='column_0', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x75f62accf890>, _spanners=Spanners([]), _heading=Heading(title='Overview', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x75f64ce9d650>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x75f64ce9e290>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=3, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=4, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x75f64ce9df50>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, categ

## Binning of the selected Predictor

The Model Report in Prediction Studio for this model will have a predictor binning plot like below.

All numbers can be derived from just the number of positives and negatives in each bin that are stored in the ADM Data Mart. The next sections will show exactly how that is done.

In [7]:
display(
    GT(
        pl.DataFrame(
            {
                "Metric": [
                    "Predictor Name",
                    "# Responses",
                    "# Bins",
                    "Selected predictor AUC (safe Pega scale)",
                ],
                "Value": [
                    predictor_name,
                    f"{predictorbinning.get_column('ResponseCount')[0]:.0f}",
                    f"{predictorbinning.get_column('BinIndex').n_unique()}",
                    f"{100 * predictorbinning.get_column('PredictorPerformance')[0]:.2f}",
                ],
            }
        )
    )
    .tab_header("Predictor information")
    .tab_options(column_labels_hidden=True)
    .tab_style(style=style.text(weight="bold"), locations=loc.body(columns="Metric"))
    .tab_options(table_margin_left=0)
)

fig = dm.plot.predictor_binning(
    model_id=modelpredictors.get_column("ModelID").unique().to_list()[0],
    predictor_name=predictor_name,
)
fig.update_layout(width=600, height=400)


GT(_tbl_data=shape: (4, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ Metric                          ┆ Value                           │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ Predictor Name                  ┆ IH.Email.Outbound.Accepted.pyH… │
│ # Responses                     ┆ 1659                            │
│ # Bins                          ┆ 5                               │
│ Selected predictor AUC (safe P… ┆ 60.40                           │
└─────────────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x75f6909ee990>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x75f62acc7a10>, _spanners=Spanners([]), _heading=Heading(title='Predictor information', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x75f62acc6e90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x75f6a00bb590>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='Metric', rows=None, mask=None), grpname=None, colname='Metric', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Metric', rows=None, mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Metric', rows=None, mask=None), grpname=None, colname='Metric', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Metric', rows=None, mask=None), grpname=None, colname='Metric', rownum=3, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x75f62acf5e50>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='0px'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='tabl

In [8]:
BinPositives = pl.col("BinPositives")
BinNegatives = pl.col("BinNegatives")
sumPositives = pl.sum("BinPositives")
sumNegatives = pl.sum("BinNegatives")

binstats = predictorbinning.select(
    pl.col("BinSymbol").alias("Range/Symbol"),
    ((BinPositives + BinNegatives) / (sumPositives + sumNegatives)).alias(
        "Responses (%)"
    ),
    BinPositives.cast(pl.Int32).alias("Positives"),
    (BinPositives / sumPositives).alias("Positives (%)"),
    BinNegatives.cast(pl.Int32).alias("Negatives"),
    (BinNegatives / sumNegatives).alias("Negatives (%)"),
    (BinPositives / (BinPositives + BinNegatives)).alias("Propensity (%)"),
    cdh_utils.z_ratio(neg_col=BinNegatives, pos_col=BinPositives),
    (
        (BinPositives / (BinPositives + BinNegatives))
        / (sumPositives / (BinPositives + BinNegatives).sum())
    ).alias("Lift"),
)
total_positives = binstats.select(pl.sum("Positives")).item()
total_negatives = binstats.select(pl.sum("Negatives")).item()
total_row = {
    "Range/Symbol": "Total",
    "Responses (%)": binstats.select(pl.sum("Responses (%)")).item(),
    "Positives": total_positives,
    "Positives (%)": binstats.select(pl.sum("Positives (%)")).item(),
    "Negatives": total_negatives,
    "Negatives (%)": binstats.select(pl.sum("Negatives (%)")).item(),
    "Propensity (%)": total_positives / (total_positives + total_negatives),
    "ZRatio": 0.0,
    "Lift": 1.0,
}

total_df = pl.DataFrame(total_row, schema=binstats.schema)
binstats = binstats.vstack(total_df)


GT(binstats).tab_header("Binning statistics").tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).fmt_percent(pl.selectors.ends_with("(%)")).fmt_number(["ZRatio", "Lift"]).tab_options(
    table_margin_left=0
).tab_options(table_margin_left=0)



GT(_tbl_data=shape: (6, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Range/Sym ┆ Responses ┆ Positives ┆ Positives ┆ … ┆ Negatives ┆ Propensit ┆ ZRatio    ┆ Lift     │
│ bol       ┆ (%)       ┆ ---       ┆ (%)       ┆   ┆ (%)       ┆ y (%)     ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ i32       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f32       ┆ f32      │
│ str       ┆ f32       ┆           ┆ f32       ┆   ┆ f32       ┆ f32       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ MISSING   ┆ 0.15009   ┆ 17        ┆ 0.082524  ┆ … ┆ 0.15967   ┆ 0.068273  ┆ -3.597362 ┆ 0.54983  │
│ <2.0      ┆ 0.229656  ┆ 30        ┆ 0.145631  ┆ … ┆ 0.241569  ┆ 0.07874   ┆ -3.550617 ┆ 0.634126 │
│ [2.0,     ┆ 0.209765  ┆ 44        ┆ 0.213592  ┆ … ┆ 0.209222  ┆ 0.126437  ┆ 0.143353  ┆ 1.018246 │
│ 2.02>     ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [2.02,    ┆ 0.332128  ┆ 89        ┆ 0.432039  ┆ … ┆ 0.317963  ┆ 0.161525  ┆ 3.115827  ┆ 1.300821 │
│ 5.02>     ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ >=5.02    ┆ 0.07836   ┆ 26        ┆ 0.126214  ┆ … ┆ 0.071576  ┆ 0.2       ┆ 2.266566  ┆ 1.61068  │
│ Total     ┆ 1.0       ┆ 206       ┆ 1.0       ┆ … ┆ 1.0       ┆ 0.124171  ┆ 0.0       ┆ 1.0      │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x75f62a930110>, _boxhead=Boxhead([ColInfo(var='Range/Symbol', type=<ColInfoTypeEnum.default: 1>, column_label='Range/Symbol', column_align='left', column_width=None), ColInfo(var='Responses (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Responses (%)', column_align='right', column_width=None), ColInfo(var='Positives', type=<ColInfoTypeEnum.default: 1>, column_label='Positives', column_align='right', column_width=None), ColInfo(var='Positives (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Positives (%)', column_align='right', column_width=None), ColInfo(var='Negatives', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives', column_align='right', column_width=None), ColInfo(var='Negatives (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives (%)', column_align='right', column_width=None), ColInfo(var='Propensity (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Propensity (%)', column_align='right', column_width=None), ColInfo(var='ZRatio', type=<ColInfoTypeEnum.default: 1>, column_label='ZRatio', column_align='right', column_width=None), ColInfo(var='Lift', type=<ColInfoTypeEnum.default: 1>, column_label='Lift', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x75f62a108c50>, _spanners=Spanners([]), _heading=Heading(title='Binning statistics', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x75f62acdfc10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x75f62a9304d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_ali

## Bin Statistics

### Positive and Negative ratios

Internally, ADM only keeps track of the total counts of positive and negative responses in each bin. Everything else is derived from those numbers. The percentages and totals are trivially derived, and the propensity is just the number of positives divided by the total. The numbers calculated here match the numbers from the datamart table exactly.

In [9]:
binning_derived = predictorbinning.select(
    pl.col("BinSymbol").alias("Range/Symbol"),
    BinPositives.alias("Positives"),
    BinNegatives.alias("Negatives"),
    ((BinPositives + BinNegatives) / (sumPositives + sumNegatives)).alias(
        "Responses %"
    ),
    (BinPositives / sumPositives).alias("Positives %"),
    (BinNegatives / sumNegatives).alias("Negatives %"),
    (BinPositives / (BinPositives + BinNegatives)).round(4).alias("Propensity"),
)

pcts = ["Responses %", "Positives %", "Negatives %", "Propensity"]
GT(binning_derived).tab_header("Derived binning statistics").tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=pcts),
).fmt_percent(pcts).tab_options(table_margin_left=0)


GT(_tbl_data=shape: (5, 7)
┌──────────────┬───────────┬───────────┬─────────────┬─────────────┬─────────────┬────────────┐
│ Range/Symbol ┆ Positives ┆ Negatives ┆ Responses % ┆ Positives % ┆ Negatives % ┆ Propensity │
│ ---          ┆ ---       ┆ ---       ┆ ---         ┆ ---         ┆ ---         ┆ ---        │
│ str          ┆ f32       ┆ f32       ┆ f32         ┆ f32         ┆ f32         ┆ f32        │
╞══════════════╪═══════════╪═══════════╪═════════════╪═════════════╪═════════════╪════════════╡
│ MISSING      ┆ 17.0      ┆ 232.0     ┆ 0.15009     ┆ 0.082524    ┆ 0.15967     ┆ 0.0683     │
│ <2.0         ┆ 30.0      ┆ 351.0     ┆ 0.229656    ┆ 0.145631    ┆ 0.241569    ┆ 0.0787     │
│ [2.0, 2.02>  ┆ 44.0      ┆ 304.0     ┆ 0.209765    ┆ 0.213592    ┆ 0.209222    ┆ 0.1264     │
│ [2.02, 5.02> ┆ 89.0      ┆ 462.0     ┆ 0.332128    ┆ 0.432039    ┆ 0.317963    ┆ 0.1615     │
│ >=5.02       ┆ 26.0      ┆ 104.0     ┆ 0.07836     ┆ 0.126214    ┆ 0.071576    ┆ 0.2        │
└──────────────┴───────────┴───────────┴─────────────┴─────────────┴─────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x75f62a931090>, _boxhead=Boxhead([ColInfo(var='Range/Symbol', type=<ColInfoTypeEnum.default: 1>, column_label='Range/Symbol', column_align='left', column_width=None), ColInfo(var='Positives', type=<ColInfoTypeEnum.default: 1>, column_label='Positives', column_align='right', column_width=None), ColInfo(var='Negatives', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives', column_align='right', column_width=None), ColInfo(var='Responses %', type=<ColInfoTypeEnum.default: 1>, column_label='Responses %', column_align='right', column_width=None), ColInfo(var='Positives %', type=<ColInfoTypeEnum.default: 1>, column_label='Positives %', column_align='right', column_width=None), ColInfo(var='Negatives %', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives %', column_align='right', column_width=None), ColInfo(var='Propensity', type=<ColInfoTypeEnum.default: 1>, column_label='Propensity', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x75f62a0eec50>, _spanners=Spanners([]), _heading=Heading(title='Derived binning statistics', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x75f62acde790>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x75f62a113f90>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=3, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=4, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=Non

### Lift

Lift is the ratio of the propensity in a particular bin over the average propensity. So a value of 1 is the average, larger than 1 means higher propensity, smaller means lower propensity:

In [10]:
positives = pl.col("Positives")
negatives = pl.col("Negatives")
sumPositives = pl.sum("Positives")
sumNegatives = pl.sum("Negatives")
GT(
    binning_derived.select(
        "Range/Symbol",
        "Positives",
        "Negatives",
        (
            (positives / (positives + negatives))
            / (sumPositives / (positives + negatives).sum())
        )
        .round(4)
        .alias("Lift"),
    )
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["Lift"])
).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Lift
MISSING,17.0,232.0,0.5498
<2.0,30.0,351.0,0.6341
"[2.0, 2.02>",44.0,304.0,1.0182
"[2.02, 5.02>",89.0,462.0,1.3008
>=5.02,26.0,104.0,1.6107


### Z-Ratio

The Z-Ratio is also a measure of the how the propensity in a bin differs from the average, but takes into account the size of the bin and thus is statistically more relevant. It represents the number of standard deviations from the average, so centers around 0. The wider the spread, the better the predictor is.
$$\frac{\text{posFraction} - \text{negFraction}}{\sqrt{\frac{\text{posFraction}(1 - \text{posFraction})}{\sum \text{positives}} + \frac{\text{negFraction}(1 - \text{negFraction})}{\sum \text{negatives}}}}$$ 

See the calculation here, which is also included in [cdh_utils' z_ratio()](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.z_ratio).

In [11]:
def z_ratio(
    pos_col: pl.Expr = pl.col("BinPositives"), neg_col: pl.Expr = pl.col("BinNegatives")
) -> pl.Expr:
    def get_fracs(pos_col=pl.col("BinPositives"), neg_col=pl.col("BinNegatives")):
        return pos_col / pos_col.sum(), neg_col / neg_col.sum()

    def z_ratio_impl(
        pos_fraction_col=pl.col("posFraction"),
        neg_fraction_col=pl.col("negFraction"),
        positives_col=pl.sum("BinPositives"),
        negatives_col=pl.sum("BinNegatives"),
    ):
        return (
            (pos_fraction_col - neg_fraction_col)
            / (
                (pos_fraction_col * (1 - pos_fraction_col) / positives_col)
                + (neg_fraction_col * (1 - neg_fraction_col) / negatives_col)
            ).sqrt()
        ).alias("ZRatio")

    return z_ratio_impl(*get_fracs(pos_col, neg_col), pos_col.sum(), neg_col.sum())


GT(
    binning_derived.select(
        "Range/Symbol", "Positives", "Negatives", "Positives %", "Negatives %"
    ).with_columns(z_ratio(positives, negatives).round(4))
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["ZRatio"])
).fmt_percent(pl.selectors.ends_with("%")).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Positives %,Negatives %,ZRatio
MISSING,17.0,232.0,8.25%,15.97%,-3.5974
<2.0,30.0,351.0,14.56%,24.16%,-3.5506
"[2.0, 2.02>",44.0,304.0,21.36%,20.92%,0.1434
"[2.02, 5.02>",89.0,462.0,43.20%,31.80%,3.1158
>=5.02,26.0,104.0,12.62%,7.16%,2.2666


## Predictor AUC


The predictor AUC is the **univariate** performance of the selected predictor against the outcome. It answers: if we rank cases using only this predictor's bins, how well does that single predictor separate positives from negatives?

This is different from the model AUC below. The model AUC is calculated from the **Classifier** bins after all active predictor contributions have been combined into the final Naive Bayes score.

This predictor-level value can be derived from the positives and negatives in the predictor bins. Because this is the first AUC calculation in the notebook, we also calculate its confidence interval from those same binned counts. The helper prints the raw 0-to-1 AUC first, then the safe Pega 50-to-100 value and its safe confidence interval.

The AUC point estimate and confidence interval are implemented in cdh_utils: [cdh_utils.auc_from_bincounts()](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.auc_from_bincounts) and [cdh_utils.auc_ci_from_bincounts()](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.auc_ci_from_bincounts).


In [12]:
predictor_auc_ci = cdh_utils.auc_ci_from_bincounts(
    pos=binning_derived.get_column("Positives"),
    neg=binning_derived.get_column("Negatives"),
    probs=binning_derived.get_column("Propensity"),
    confidence_level=0.95,
)
predictor_performance = predictorbinning.get_column("PredictorPerformance")[0]

if predictor_auc_ci["ci_available"]:
    safe_predictor_ci = f"[{100 * predictor_auc_ci['safe_ci_lower']:.2f}, {100 * predictor_auc_ci['safe_ci_upper']:.2f}]"
else:
    safe_predictor_ci = f"unavailable ({predictor_auc_ci['ci_reason']})"

print(
    f"Selected predictor AUC from predictor bins: {predictor_auc_ci['auc']:.6f} | "
    f"safe Pega scale: {100 * predictor_auc_ci['auc']:.2f} | "
    f"95% safe CI: {safe_predictor_ci}"
)
print(
    f"Predictor Performance column: {predictor_performance:.6f} | "
    f"safe Pega scale: {100 * predictor_performance:.2f}"
)


Selected predictor AUC from predictor bins: 0.609602 | safe Pega scale: 60.96 | 95% safe CI: [57.11, 64.81]
Predictor Performance column: 0.603987 | safe Pega scale: 60.40


## Model AUC

The model's overall AUC — shown as **Model Performance (AUC)** in the model overview above — is derived from the **Classifier** row, not from the selected predictor and not as an average of the predictor AUCs.

The Classifier is the PAVA-calibrated score distribution: after all predictor log-odds contributions are summed into the final score, the Classifier records the observed positive and negative response counts per score interval. Because those bins carry the same $pos$ / $neg$ structure as any predictor bin, the model AUC and confidence interval can be calculated from the Classifier bins with the same [`auc_ci_from_bincounts()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.auc_ci_from_bincounts) helper used above.

$$AUC_{\text{model}} = \text{auc\_from\_bincounts}(\,pos_{\text{classifier}},\; neg_{\text{classifier}}\,)$$

> **Portfolio AUC:** Each Naïve Bayes model covers one action / channel / direction combination. The natural portfolio-level summary is a response-count-weighted average of the individual model AUCs — which is what `cdh_utils.weighted_performance_polars()` computes across a set of models. The same principle applies to Adaptive Gradient Boosting models, where each action has its own independently validated AUC.


In [13]:
classifier_bins = (
    modelpredictors
    .filter(pl.col("EntryType") == "Classifier")
    .sort("BinIndex")
)

classifier_pos = classifier_bins.get_column("BinPositives").to_numpy()
classifier_neg = classifier_bins.get_column("BinNegatives").to_numpy()
classifier_scores = classifier_bins.get_column("BinIndex").to_numpy()
classifier_order = np.argsort(classifier_scores)[::-1]

FPR = np.cumsum(classifier_neg[classifier_order]) / np.sum(classifier_neg[classifier_order])
TPR = np.cumsum(classifier_pos[classifier_order]) / np.sum(classifier_pos[classifier_order])
TPR = np.insert(TPR, 0, 0, axis=0)
FPR = np.insert(FPR, 0, 0, axis=0)

model_auc_ci = cdh_utils.auc_ci_from_bincounts(
    pos=classifier_bins.get_column("BinPositives"),
    neg=classifier_bins.get_column("BinNegatives"),
    probs=classifier_bins.get_column("BinIndex"),
    confidence_level=0.95,
)
model_performance = modelpredictors.select(pl.col("Performance").unique()).item()

fig = px.line(
    x=FPR,
    y=TPR,
    labels=dict(x="False positive rate", y="Sensitivity"),
    title=f"Overall model AUC = {model_auc_ci['auc']:.3f}",
    width=525,
    height=525,
    range_x=[0, 1],
    range_y=[0, 1],
    template="none",
)
fig.update_traces(
    fill="tozeroy",
    fillcolor="rgba(0, 31, 95, 0.15)",
    line=dict(color="#001F5F", width=3),
    marker=dict(size=7),
)
fig.add_shape(
    type="line",
    line=dict(color="#63666F", dash="dash"),
    x0=0,
    x1=1,
    y0=0,
    y1=1,
    layer="above",
)
fig.show()

if model_auc_ci["ci_available"]:
    safe_model_ci = f"[{100 * model_auc_ci['safe_ci_lower']:.2f}, {100 * model_auc_ci['safe_ci_upper']:.2f}]"
else:
    safe_model_ci = f"unavailable ({model_auc_ci['ci_reason']})"

print(
    f"Overall model AUC from Classifier bins: {model_auc_ci['auc']:.6f} | "
    f"safe Pega scale: {100 * model_auc_ci['auc']:.2f} | "
    f"95% safe CI: {safe_model_ci}"
)
print(
    f"Model Performance column: {model_performance:.6f} | "
    f"safe Pega scale: {100 * model_performance:.2f}"
)


Overall model AUC from Classifier bins: 0.781583 | safe Pega scale: 78.16 | 95% safe CI: [74.54, 81.77]
Model Performance column: 0.774901 | safe Pega scale: 77.49


## Predictor Contributions

### Naive Bayes and Log Odds

The basis for the Naive Bayes algorithm is Bayes' Theorem:

$$\Pr(C_k \mid x) = \frac{\Pr(x \mid C_k)\Pr(C_k)}{\Pr(x)}$$

with $C_k$ the outcome and $x$ the customer. Bayes' theorem turns the
question "what's the probability to accept this action given a customer" around to
"what's the probability of this customer given an action". With the independence
assumption, and after applying a log odds transformation we get a log odds score
that can be calculated efficiently and in a numerically stable manner:

$$\text{log odds score} = \sum_{p \in \text{active predictors}} \log(\Pr(x_p \mid \text{Positive})) + \log(p_{\text{positive}}) - \sum_p \log(\Pr(x_p \mid \text{Negative})) - \log(p_{\text{negative}})$$
note that the _prior_ can be written as:

$$\log(p_{\text{positive}}) - \log(p_{\text{negative}}) = \log\left(\frac{\text{TotalPositives}}{\text{Total}}\right) - \log\left(\frac{\text{TotalNegatives}}{\text{Total}}\right) = \log(\text{TotalPositives}) - \log(\text{TotalNegatives})$$


### Predictor Contribution

The contribution (_conditional log odds_) of an active predictor $p$ for bin $i$ with the number
of positive and negative responses in $\text{Positives}_i$ and $\text{Negatives}_i$ is calculated as (note the "Laplace smoothing" to avoid log 0 issues):

$$\text{contribution}_p = \log\left(\text{Positives}_i + \frac{1}{\text{nBins}}\right) - \log\left(\text{Negatives}_i + \frac{1}{\text{nBins}}\right) - \log\left(1 + \sum_{i = 1}^{\text{nBins}} \text{Positives}_i\right) + \log\left(1 + \sum_i \text{Negatives}_i\right)$$


In [14]:
N = binning_derived.shape[0]
GT(
    binning_derived.with_columns(
        LogOdds=(pl.col("Positives %") / pl.col("Negatives %")).log().round(5),
        ModifiedLogOdds=(
            ((positives + 1 / N).log() - (positives.sum() + 1).log())
            - ((negatives + 1 / N).log() - (negatives.sum() + 1).log())
        ).round(5),
    ).drop("Responses %", "Propensity")
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=["LogOdds", "ModifiedLogOdds"]),
).fmt_percent(pl.selectors.ends_with("%")).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Positives %,Negatives %,LogOdds,ModifiedLogOdds
MISSING,17.0,232.0,8.25%,15.97%,-0.66001,-0.65334
<2.0,30.0,351.0,14.56%,24.16%,-0.50608,-0.50416
"[2.0, 2.02>",44.0,304.0,21.36%,20.92%,0.02067,0.02039
"[2.02, 5.02>",89.0,462.0,43.20%,31.80%,0.30658,0.30424
>=5.02,26.0,104.0,12.62%,7.16%,0.56722,0.5688


## Propensity mapping

### Log odds contribution for all the predictors

$$\text{score} = \frac{\log(1 + \text{TotalPositives}) - \log(1 + \text{TotalNegatives}) + \sum_p \text{contribution}_p}{1 + \text{nActivePredictors}}$$

Here, $\text{TotalPositives}$ and $\text{TotalNegatives}$ are the total number of
positive and negative responses in the model, and $\text{contribution}_p$ is
the conditional log odds contribution of each active predictor.


In [15]:
def middle_bin():
    return pl.col("BinIndex") == (pl.max("BinIndex") / 2).floor().cast(pl.UInt32)


if not all(
    col in modelpredictors.columns for col in ["BinLowerBound", "BinUpperBound"]
):

    def extract_numbers_in_contents(s: str, index):
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", s)
        try:
            number = float(numbers[index])
        except:
            number = 0
        return number

    modelpredictors = modelpredictors.with_columns(
        pl.col("Contents").cast(pl.Utf8)
    ).with_columns(
        pl.when(pl.col("Type") == "numeric")
        .then(
            pl.col("Contents").map_elements(
                lambda col: extract_numbers_in_contents(col, 0)
            )
        )
        .otherwise(pl.lit(-9999))
        .alias("BinLowerBound")
        .cast(pl.Float32),
        pl.when(pl.col("Type") == "numeric")
        .then(
            pl.col("Contents").map_elements(
                lambda col: extract_numbers_in_contents(col, 1)
            )
        )
        .otherwise(pl.lit(-9999))
        .alias("BinUpperBound")
        .cast(pl.Float32),
    )


def row_wise_log_odds(bin, positives, negatives):
    bin, N = bin.list.get(0) - 1, positives.list.len()
    pos, neg = positives.list.get(bin), negatives.list.get(bin)
    pos_sum, neg_sum = positives.list.sum(), negatives.list.sum()
    return (
        ((pos + (1 / N)).log() - (pos_sum + 1).log())
        - (((neg + (1 / N)).log()) - (neg_sum + 1).log())
    ).alias("Modified Log odds")


df = (
    modelpredictors.filter(pl.col("PredictorName") != "Classifier")
    .group_by("PredictorName")
    .agg(
        Value=pl.when(pl.col("Type").first() == "numeric")
        .then(
            ((pl.col("BinLowerBound") + pl.col("BinUpperBound")) / 2).filter(
                middle_bin()
            )
        )
        .otherwise(
            pl.col("BinSymbol").str.split(",").list.first().filter(middle_bin())
        ),
        Bin=pl.col("BinIndex").filter(middle_bin()),
        Positives=pl.col("BinPositives"),
        Negatives=pl.col("BinNegatives"),
    )
    .with_columns(
        pl.col(["Positives", "Negatives"]).list.get(pl.col("Bin").list.get(0) - 1),
        pl.col("Bin", "Value").list.get(0),
        LogOdds=row_wise_log_odds(
            pl.col("Bin"), pl.col("Positives"), pl.col("Negatives")
        ).round(4),
    )
    .sort("PredictorName")
)

classifier = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .with_columns(
        Propensity=(BinPositives / (BinPositives / BinNegatives)),
        FinalPropensity=((0.5 + BinPositives) / (1 + BinPositives + BinNegatives)),
        ZRatio=cdh_utils.z_ratio(neg_col=BinNegatives, pos_col=BinPositives).round(4),
        Lift=(
            (BinPositives / (BinPositives + BinNegatives))
            / (sumPositives / (BinPositives + BinNegatives).sum())
        ),
    )
    .select(
        [
            pl.col("BinIndex").alias("Index"),
            pl.col("BinSymbol").alias("Bin"),
            BinPositives.alias("Positives"),
            BinNegatives.alias("Negatives"),
            (pl.cum_sum("BinResponseCount") / pl.sum("BinResponseCount")).alias(
                "Cum. Total (%)"
            ),
            (pl.col("BinPropensity")).alias("Propensity (%)"),
            (pl.col("FinalPropensity")).alias("Final Propensity (%)"),
            (pl.cum_sum("BinPositives") / pl.sum("BinPositives")).alias(
                "Cum Positives (%)"
            ),
            pl.col("ZRatio"),
            pl.col("Lift").alias("Lift(%)"),
            pl.col("BinResponseCount").alias("Responses"),
        ]
    )
)
classifierLogOffset = log(1 + classifier["Positives"].sum()) - log(
    1 + classifier["Negatives"].sum()
)

propensity_mapping = df.vstack(
    pl.DataFrame(
        dict(
            zip(
                df.columns,
                ["Final Score"]
                + [None] * 4
                + [(df["LogOdds"].sum() + classifierLogOffset) / (len(df) + 1)],
            )
        ),
        schema=df.schema,
    )
).with_columns(pl.col("LogOdds").round(4))

GT(propensity_mapping).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="PredictorName")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["LogOdds"])
).tab_options(table_margin_left=0)

PredictorName,Value,Bin,Positives,Negatives,LogOdds
Customer.Age,34.56,4,9.0,198.0,-1.13
Customer.AnnualIncome,-24043.049,1,74.0,1189.0,-0.8232
Customer.BusinessSegment,middleSegmentPlus,1,96.0,993.0,-0.3839
Customer.CLV,NON-MISSING,1,111.0,593.0,0.2773
Customer.CLV_VALUE,1345.52,4,31.0,297.0,-0.3068
Customer.CreditScore,518.92,3,33.0,205.0,0.1265
Customer.Date_of_Birth,18773.504,5,28.0,152.0,0.2606
Customer.Gender,U,1,52.0,504.0,-0.3162
Customer.MaritalStatus,No Resp+,1,67.0,768.0,-0.4852
Customer.NetWealth,17992.398,4,51.0,179.0,0.6955


### From Log Odds Score to Final Propensity

The log odds score is mapped to a propensity using a score distribution, which is built using the [PAV(A)](https://en.wikipedia.org/wiki/Isotonic_regression) (Pool Adjacent Violators Algorithm), a form of isotonic regression that ensures the mapping is monotonically increasing.

For each bin (score interval) the final propensity is calculated from the observed positive and negative counts:

$$Final\ Propensity = \frac{0.5 + BinPositives}{1 + BinPositives + BinNegatives}$$

The addition of 0.5 is for Laplace smoothing. An empty model (no responses yet) returns a neutral propensity of 0.5, rather than 0 or undefined. This quickly converges to the observed success rate as more responses are received.

The transformation is a calibrated, isotonic mapping from log odds to propensity, grounded in the actual observed response rates of the model.


In [16]:
# TODO see if we can port the "getActiveRanges" code to python so to highlight the classifier rows that are "active"

GT(classifier.drop("Responses")).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Index")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=["Final Propensity (%)"]),
).fmt_percent(pl.selectors.ends_with("(%)")).tab_options(table_margin_left=0)

Index,Bin,Positives,Negatives,Cum. Total (%),Propensity (%),Final Propensity (%),Cum Positives (%),ZRatio,Lift(%)
1,<-0.21,17.0,466.0,29.11%,3.52%,3.62%,8.25%,-10.4708,1.89%
2,"[-0.21, -0.185>",8.0,133.0,37.61%,5.67%,5.99%,12.14%,-3.4129,3.05%
3,"[-0.185, -0.175>",3.0,48.0,40.69%,5.88%,6.73%,13.59%,-1.9295,3.16%
4,"[-0.175, -0.105>",28.0,370.0,64.68%,7.04%,7.14%,27.18%,-4.4849,3.78%
5,"[-0.105, -0.095>",4.0,51.0,67.99%,7.27%,8.04%,29.13%,-1.4577,3.90%
6,"[-0.095, -0.09>",2.0,19.0,69.26%,9.52%,11.36%,30.10%,-0.4518,5.11%
7,"[-0.09, -0.065>",9.0,77.0,74.44%,10.47%,10.92%,34.47%,-0.6039,5.62%
8,"[-0.065, -0.02>",30.0,154.0,85.53%,16.30%,16.49%,49.03%,1.5325,8.75%
9,"[-0.02, 0.03>",37.0,65.0,91.68%,36.27%,36.41%,66.99%,4.9425,19.48%
10,"[0.03, 0.06>",20.0,29.0,94.64%,40.82%,41.00%,76.70%,3.6811,21.91%


The chart below visualizes the score distribution from the table above. The score bin that contains the calculated final score for the example customer is highlighted.

In [17]:
score = propensity_mapping.filter(PredictorName="Final Score")["LogOdds"][0]
score_bin = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .select(
        pl.col("BinSymbol").filter(
            pl.lit(score).is_between(pl.col("BinLowerBound"), pl.col("BinUpperBound"))
        )
    )
    .item()
)
score_responses = modelpredictors.filter(
    (pl.col("EntryType") == "Classifier") & (pl.col("BinSymbol") == score_bin)
)["BinResponseCount"][0]

score_bin_index = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")["BinSymbol"]
    .to_list()
    .index(score_bin)
)

score_propensity = classifier.row(score_bin_index, named=True)[
    "Final Propensity (%)"
]

final_propensity = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .with_columns(
        FinalPropensity=(
            (0.5 + BinPositives) / (1 + BinPositives + BinNegatives)
        ).round(5),
    )
    .select(
        pl.col("FinalPropensity").filter(
            (pl.col("BinLowerBound") < score) & (pl.col("BinUpperBound") > score)
        )
    )["FinalPropensity"][0]
)
fig = dm.plot.score_distribution(
    model_id=modelpredictors.get_column("ModelID").unique().to_list()[0]
).add_annotation(
    x=score_bin,
    y=score_propensity / 100,
    text=f"Returned propensity: {score_propensity*100:.2f}%",
    bgcolor="#FFFFFF",
    bordercolor="#000000",
    showarrow=False,
    yref="y2",
    opacity=0.7,
)
bin_index = list(fig.data[0]["x"]).index(score_bin)
fig.data[0]["marker_color"] = (
    ["grey"] * bin_index
    + ["#1f77b4"]
    + ["grey"] * (classifier.shape[0] - bin_index - 1)
)
fig

## Feature Importance

Feature importance in Naive Bayes models represents **how strongly a predictor differentiates between positive and negative outcomes**. It measures the average magnitude of log odds contributions across the predictor's bins.

A predictor with high importance has bins with strong log odds values (far from zero), indicating strong predictive power. A predictor with low importance has bins with weak log odds values (close to zero), indicating weak predictive power.

### Formula

For a predictor with bins $i = 1, \ldots, n$:

**Step 1: Log odds per bin** (with Laplace smoothing $\frac{1}{n}$):

$$\text{LogOdds}_i = \log\left(\text{Pos}_i + \frac{1}{n}\right) - \log\left(\sum_i \text{Pos}_i + 1\right) - \left[\log\left(\text{Neg}_i + \frac{1}{n}\right) - \log\left(\sum_i \text{Neg}_i + 1\right)\right]$$

**Step 2: Absolute log odds per bin**:

$$|\text{LogOdds}_i|$$

**Step 3: Feature importance** (weighted average of absolute log odds):

$$\text{Importance} = \frac{\sum_i |\text{LogOdds}_i| \times \text{Responses}_i}{\sum_i \text{Responses}_i}$$

**Step 4: Optional scaling** (default):

$$\text{Scaled Importance} = \frac{\text{Importance}}{\max(\text{Importance})} \times 100$$

### Interpretation

- **Higher values**: Predictor has strong log odds values (strong predictive power)
- **Lower values**: Predictor has weak log odds values (weak predictive power)
- **Zero**: All bins have zero log odds (predictor adds no information)

The absolute log odds captures the magnitude of each bin's contribution to the model score, weighted by how many responses are in that bin. This matches the Pega platform implementation.


In [18]:
# Calculate feature importance for all active predictors
predictor_importance = (
    modelpredictors.filter(pl.col("PredictorName") != "Classifier")
    .with_columns(
        # Calculate both unscaled and scaled importance
        cdh_utils.feature_importance(scaled=False).alias("Importance"),
        cdh_utils.feature_importance(scaled=True).alias("ScaledImportance"),
    )
    .group_by("PredictorName", "ModelID", "PredictorCategory")
    .agg(pl.first("Importance"), pl.first("ScaledImportance"))
    .sort("ScaledImportance", descending=True)
)

GT(predictor_importance.drop("ModelID")).tab_header("Feature Importance by Predictor").fmt_number(
    "Importance", decimals=4
).fmt_number("ScaledImportance", decimals=2)


GT(_tbl_data=shape: (36, 4)
┌─────────────────────────────────┬───────────────────┬────────────┬──────────────────┐
│ PredictorName                   ┆ PredictorCategory ┆ Importance ┆ ScaledImportance │
│ ---                             ┆ ---               ┆ ---        ┆ ---              │
│ str                             ┆ str               ┆ f64        ┆ f64              │
╞═════════════════════════════════╪═══════════════════╪════════════╪══════════════════╡
│ Customer.AnnualIncome           ┆ Customer          ┆ 0.906914   ┆ 100.0            │
│ Customer.NetWealth              ┆ Customer          ┆ 0.860623   ┆ 94.895815        │
│ Customer.WinScore               ┆ Customer          ┆ 0.666106   ┆ 73.44758         │
│ Customer.CreditScore            ┆ Customer          ┆ 0.634783   ┆ 69.993735        │
│ Customer.RelationshipStartDate  ┆ Customer          ┆ 0.511196   ┆ 56.366584        │
│ …                               ┆ …                 ┆ …          ┆ …                │
│ IH.SMS.Outbound.Loyal.pyHistor… ┆ IH                ┆ 0.129175   ┆ 14.243341        │
│ IH.Email.Outbound.Churned.pyHi… ┆ IH                ┆ 0.127129   ┆ 14.017818        │
│ Customer.NoOfDependents         ┆ Customer          ┆ 0.122691   ┆ 13.528398        │
│ IH.SMS.Outbound.Rejected.pxLas… ┆ IH                ┆ 0.12138    ┆ 13.383844        │
│ IH.SMS.Outbound.Churned.pxLast… ┆ IH                ┆ 0.105501   ┆ 11.632982        │
└─────────────────────────────────┴───────────────────┴────────────┴──────────────────┘, _body=<great_tables._gt_data.Body object at 0x75f5bb2d6790>, _boxhead=Boxhead([ColInfo(var='PredictorName', type=<ColInfoTypeEnum.default: 1>, column_label='PredictorName', column_align='left', column_width=None), ColInfo(var='PredictorCategory', type=<ColInfoTypeEnum.default: 1>, column_label='PredictorCategory', column_align='left', column_width=None), ColInfo(var='Importance', type=<ColInfoTypeEnum.default: 1>, column_label='Importance', column_align='right', column_width=None), ColInfo(var='ScaledImportance', type=<ColInfoTypeEnum.default: 1>, column_label='ScaledImportance', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x75f5bb2c6650>, _spanners=Spanners([]), _heading=Heading(title='Feature Importance by Predictor', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x75f5bb3641d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x75f5bb38a6d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x75f5bb38b010>, _formats=[<great_tables._gt_data.FormatInfo object at 0x75f5bb38b210>, <great_tables._gt_data.FormatInfo object at 0x75f5fb474950>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal')

In [19]:
# Visualize feature importance
predictor_order = predictor_importance.get_column("PredictorName").to_list()

fig = px.bar(
    predictor_importance,
    x="PredictorName",
    y="ScaledImportance",
    color="PredictorCategory",
    color_discrete_map=dm.predictor_category_color_map,
    category_orders={"PredictorName": predictor_order},
    title="Predictor Feature Importance (Scaled 0-100)",
    labels={
        "ScaledImportance": "Importance",
        "PredictorName": "",
        "PredictorCategory": "Predictor category",
    },
    template="pega",
)
fig.update_layout(
    xaxis_tickangle=-45,
    height=500,
    margin=dict(b=180)  # Increase bottom margin for longer rotated labels
)
fig.update_xaxes(
    categoryorder="array",
    categoryarray=predictor_order,
    tickfont=dict(size=10),
)  # Smaller font for x-axis labels
fig.show()
